In [ ]:
# ─── 1. Install ───────────────────────────────────────────────────────────────
!pip install -q datasets scikit-learn

# ─── 2. Mount Drive ───────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/TFIDF_SVM_IFND"
import os, time
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── 3. Imports ───────────────────────────────────────────────────────────────
import numpy as np
import joblib
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

# ─── 4. Load dataset ──────────────────────────────────────────────────────────
print("⏳ Loading dataset...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")
print(hf_dataset)

# ─── 5. Extract text & labels ─────────────────────────────────────────────────
print("⏳ Extracting text...")
train_texts  = hf_dataset['train']['text']
train_labels = hf_dataset['train']['label']
val_texts    = hf_dataset['validation']['text']
val_labels   = hf_dataset['validation']['label']
test_texts   = hf_dataset['test']['text']
test_labels  = hf_dataset['test']['label']

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

# ─── 6. TF-IDF Vectorization ──────────────────────────────────────────────────
print("⏳ Fitting TF-IDF...")
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True
)
X_train = vectorizer.fit_transform(train_texts)
X_val   = vectorizer.transform(val_texts)
X_test  = vectorizer.transform(test_texts)
print(f"Feature matrix: {X_train.shape}")

# ─── 7. Train SVM ─────────────────────────────────────────────────────────────
print("⏳ Training SVM...")
t0  = time.perf_counter()
svm = LinearSVC(C=1.0, max_iter=2000)
clf = CalibratedClassifierCV(svm, cv=3)
clf.fit(X_train, train_labels)
print(f"✅ Done in {time.perf_counter()-t0:.1f}s")

# ─── 8. Validation ────────────────────────────────────────────────────────────
val_preds = clf.predict(X_val)
val_probs = clf.predict_proba(X_val)[:, 1]
p, r, f1, _ = precision_recall_fscore_support(
    val_labels, val_preds, average="binary")
print(f"\nValidation:")
print(f"  Accuracy : {accuracy_score(val_labels, val_preds)*100:.2f}%")
print(f"  F1       : {f1*100:.2f}%")
print(f"  AUC      : {roc_auc_score(val_labels, val_probs):.5f}")

# ─── 9. Test Evaluation ───────────────────────────────────────────────────────
print("\n📊 Evaluating on test set...")
test_preds = clf.predict(X_test)
test_probs = clf.predict_proba(X_test)[:, 1]
p, r, f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average="binary")

print(f"\n{'='*40}")
print(f"Test Accuracy : {accuracy_score(test_labels, test_preds)*100:.2f}%")
print(f"Test Precision: {p*100:.2f}%")
print(f"Test Recall   : {r*100:.2f}%")
print(f"Test F1       : {f1*100:.2f}%")
print(f"Test AUC      : {roc_auc_score(test_labels, test_probs):.5f}")
print(f"{'='*40}")

# ─── 10. Latency + Model size ─────────────────────────────────────────────────
dummy_text = ["This is a sample news headline for inference measurement."]
latencies  = []
for _ in range(100):
    t0 = time.perf_counter()
    x  = vectorizer.transform(dummy_text)
    clf.predict(x)
    latencies.append((time.perf_counter() - t0) * 1000)

latencies     = np.array(latencies)
model_path    = os.path.join(SAVE_DIR, "tfidf_svm.joblib")
joblib.dump({"vectorizer": vectorizer, "clf": clf}, model_path)
model_size_mb = os.path.getsize(model_path) / 1024**2
n_params      = X_train.shape[1] * 2

print(f"\n{'='*40}")
print(f"Total params    : {n_params:,}  (TF-IDF features x classes)")
print(f"Model size      : {model_size_mb:.1f} MB")
print(f"Latency median  : {np.median(latencies):.2f} ms")
print(f"VRAM            : 0 MB  (CPU-based)")
print(f"{'='*40}")
print(f"✅ Saved: {model_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/8416 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1052 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1053 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 8416
    })
    validation: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1052
    })
    test: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1053
    })
})
⏳ Extracting text...
Train: 8416 | Val: 1052 | Test: 1053
⏳ Fitting TF-IDF...
Feature matrix: (8416, 10000)
⏳ Training SVM...
✅ Done in 0.1s

Validation:
  Accuracy : 98.38%
  F1       : 98.95%
  AUC      : 0.99275

📊 Evaluating on test set...

Test Accuracy : 98.39%
Test Precision: 98.77%
Test Recall   : 99.13%
Test F1       : 98.95%
Test AUC      : 0.99731

Total params    : 20,000  (TF-IDF features x classes)
Model size      : 0.6 MB
Latency median  : 2.45 ms
VRAM            : 0 MB  (CPU-based)
✅ Saved: /content/drive/MyDrive/TFIDF_SVM_IFND/tfidf_svm.joblib
